# PEFT Transfer Training: HeartBERT, ECG-PT, HuBERT-ECG

Three pretrained ECG foundation models fine-tuned on the PTB-XL 5-class diagnostic
classification task using **parameter-efficient fine-tuning (PEFT)** only.
No full fine-tuning is performed.

| Model | Architecture | HuggingFace ID | Input |
|---|---|---|---|
| HeartBERT | RoBERTa encoder | `Bayesiano/HeartBERT` | Lead II → quantised text (32 bins) |
| ECG-PT | GPT-2 decoder | `Tconnector/ecg-pt` | Lead II → patch tokens |
| HuBERT-ECG | HuBERT encoder | `Edoardo-BS/hubert-ecg-base` | 12-lead float tensor |

> **Note:** `Bayesiano/HeartBERT` and `Tconnector/ecg-pt` are not publicly available on HuggingFace.
> HeartBERT weights are downloaded automatically from Google Drive (cached to `~/.cache/heartbert/`).
> ECG-PT falls back to plain GPT-2 (same architecture, general-language weights).

Each model is trained twice:
- **LoRA r=16** — low-rank adapter in attention Q/K/V projections (~1–2% of params)
- **DoRA r=16** — weight-decomposed LoRA; separates magnitude and direction updates

Training: **25 epochs**, AdamW lr=1.5e-4, lora_alpha=32. Results saved to `results/<experiment_name>/`.

In [1]:
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '0'   # allow HF download on first run
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import torch
import wfdb

from src.utils.config import CFG
from src.preprocessing.label_utils import load_all_labels, SUPERCLASSES

DATA_PATH    = CFG['data']['path']
RESULTS_PATH = CFG['paths']['results']
HUBERT_SIZE  = CFG['model']['hubert_size']   # "base"
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device       : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Results dir  : {RESULTS_PATH}')
print(f'HuBERT size  : {HUBERT_SIZE}')

Device       : cuda
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM         : 8.6 GB
Results dir  : D:\GitHub\biosignal-xai\results/
HuBERT size  : base


## 2. Data

PTB-XL folds: 1–8 train, 9 val, 10 test (never touched here).

**Two loading paths:**
- `X_lead` / `y_*` — Lead II (index 1) as numpy arrays for HeartBERT and ECG-PT.
  Loaded upfront because HuggingFace tokenisers cannot run inside DataLoader workers.
- `ECGDatasetFull` — 12-lead on-the-fly loader for HuBERT-ECG (avoids ~800 MB preload).

In [2]:
from src.preprocessing.dataset_full import ECGDatasetFull

Y = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')

train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]

# ── Single-lead numpy arrays (HeartBERT / ECG-PT) ─────────────────────────
LEAD_IDX = 1   # Lead II

def _load_lead(df, data_path, lead_idx=LEAD_IDX):
    X, y = [], []
    for i, (_, row) in enumerate(df.iterrows()):
        sig, _ = wfdb.rdsamp(data_path + row['filename_lr'])
        X.append(sig[:, lead_idx].astype(np.float32))
        y.append(np.array(row['label_vec'], dtype=np.float32))
        if (i + 1) % 2000 == 0:
            print(f"  Loaded {i+1}/{len(df)} records...")
    return np.stack(X), np.stack(y)

print("Loading single-lead arrays...")
X_train_lead, y_train = _load_lead(train_df, DATA_PATH)
X_val_lead,   y_val   = _load_lead(val_df,   DATA_PATH)

# ── Full 12-lead datasets (HuBERT-ECG) ────────────────────────────────────
train_ds_full = ECGDatasetFull(train_df, DATA_PATH)
val_ds_full   = ECGDatasetFull(val_df,   DATA_PATH)

print(f'\nSingle-lead arrays:')
print(f'  X_train_lead : {X_train_lead.shape}  y_train : {y_train.shape}')
print(f'  X_val_lead   : {X_val_lead.shape}  y_val   : {y_val.shape}')
print(f'\nFull-lead datasets:')
print(f'  train_ds_full : {len(train_ds_full):,} records')
print(f'  val_ds_full   : {len(val_ds_full):,} records')

Records with valid labels: 21375
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5108 (23.9%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Loading single-lead arrays...
  Loaded 2000/17073 records...
  Loaded 4000/17073 records...
  Loaded 6000/17073 records...
  Loaded 8000/17073 records...
  Loaded 10000/17073 records...
  Loaded 12000/17073 records...
  Loaded 14000/17073 records...
  Loaded 16000/17073 records...
  Loaded 2000/2145 records...

Single-lead arrays:
  X_train_lead : (17073, 1000)  y_train : (17073, 5)
  X_val_lead   : (2145, 1000)  y_val   : (2145, 5)

Full-lead datasets:
  train_ds_full : 17,073 records
  val_ds_full   : 2,145 records


## 3. HeartBERT

RoBERTa encoder pretrained on ECG-as-text. Weights are loaded from Google Drive
(`~/.cache/heartbert/` after first download). Input ECG signals are quantised into
**32-bin letter strings** (A-Z a-f) before tokenisation — up from 20 bins for better signal fidelity.

Each run:
1. Fresh `HeartBERTClassifier` instance; weights downloaded from Google Drive on first run
2. PEFT adapters attached (LoRA or DoRA, **r=16 alpha=32**)
3. Sanity check: output shape `(2, 5)`, trainable < 5 %
4. `fit()` — 25 epochs, AdamW lr=1.5e-4, BCEWithLogitsLoss, saves best checkpoint

In [3]:
from src.models.heartbert import HeartBERTClassifier

_probe = HeartBERTClassifier(num_labels=5)
_probe.load()
_probe.apply_peft(use_dora=False)

# Shape check: single dummy signal
_dummy_X = np.random.randn(2, 1000).astype(np.float32)
_dummy_p = _probe.predict(_dummy_X)
assert _dummy_p.shape == (2, 5), f"Shape error: {_dummy_p.shape}"

_params = _probe.count_parameters()
assert _params['trainable'] / _params['total'] < 0.05, \
    f"Trainable fraction {_params['trainable']/_params['total']:.1%} exceeds 5%"

print(f"Shape OK : (2, 1000) -> {_dummy_p.shape}")
print(f"Params   : {_params['trainable']:,} / {_params['total']:,} = {_params['percentage']}")
del _probe

HeartBERT weights found in cache: C:\Users\Teodora\.cache\heartbert
Loading HeartBERT from C:\Users\Teodora\.cache\heartbert ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at C:\Users\Teodora\.cache\heartbert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 889,349 || all params: 84,344,074 || trainable%: 1.0544
Shape OK : (2, 1000) -> (2, 5)
Params   : 889,349 / 84,344,074 = 1.1%


In [4]:
from src.models.heartbert import HeartBERTClassifier

model_hb_lora = HeartBERTClassifier(num_labels=5)
model_hb_lora.load()
model_hb_lora.apply_peft(use_dora=False)

auc_hb_lora, hist_hb_lora = model_hb_lora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'heartbert_lora_r16',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_hb_lora
import gc; gc.collect()
torch.cuda.empty_cache()

HeartBERT weights found in cache: C:\Users\Teodora\.cache\heartbert
Loading HeartBERT from C:\Users\Teodora\.cache\heartbert ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at C:\Users\Teodora\.cache\heartbert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 889,349 || all params: 84,344,074 || trainable%: 1.0544
  Tokenising signals (this takes ~1 min for 17k records)...

 Experiment : heartbert_lora_r16
 Device     : cuda
 Trainable  : 889,349  (1.1%)

Epoch 01/25  train=0.4992  val=0.4795  AUC=0.7149
  * Best saved — AUC 0.7149
Epoch 02/25  train=0.4774  val=0.4729  AUC=0.7239
  * Best saved — AUC 0.7239
Epoch 03/25  train=0.4680  val=0.4701  AUC=0.7384
  * Best saved — AUC 0.7384
Epoch 04/25  train=0.4595  val=0.4645  AUC=0.7474
  * Best saved — AUC 0.7474
Epoch 05/25  train=0.4530  val=0.4631  AUC=0.7480
  * Best saved — AUC 0.7480
Epoch 06/25  train=0.4444  val=0.4552  AUC=0.7574
  * Best saved — AUC 0.7574
Epoch 07/25  train=0.4390  val=0.4553  AUC=0.7561
Epoch 08/25  train=0.4338  val=0.4478  AUC=0.7627
  * Best saved — AUC 0.7627
Epoch 09/25  train=0.4294  val=0.4532  AUC=0.7638
  * Best saved — AUC 0.7638
Epoch 10/25  train=0.4237  val=0.4554  AUC=0.7645
  * Best saved — AUC 0.7645
Epoch 11/25  train=0.4

In [5]:
from src.models.heartbert import HeartBERTClassifier

model_hb_dora = HeartBERTClassifier(num_labels=5)
model_hb_dora.load()
model_hb_dora.apply_peft(use_dora=True)

auc_hb_dora, hist_hb_dora = model_hb_dora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'heartbert_dora_r16',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_hb_dora
import gc; gc.collect()
torch.cuda.empty_cache()

HeartBERT weights found in cache: C:\Users\Teodora\.cache\heartbert
Loading HeartBERT from C:\Users\Teodora\.cache\heartbert ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at C:\Users\Teodora\.cache\heartbert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 898,565 || all params: 84,353,290 || trainable%: 1.0652
  Tokenising signals (this takes ~1 min for 17k records)...

 Experiment : heartbert_dora_r16
 Device     : cuda
 Trainable  : 898,565  (1.1%)

Epoch 01/25  train=0.4986  val=0.4989  AUC=0.7100
  * Best saved — AUC 0.7100
Epoch 02/25  train=0.4759  val=0.4723  AUC=0.7229
  * Best saved — AUC 0.7229
Epoch 03/25  train=0.4667  val=0.4738  AUC=0.7324
  * Best saved — AUC 0.7324
Epoch 04/25  train=0.4581  val=0.4735  AUC=0.7364
  * Best saved — AUC 0.7364
Epoch 05/25  train=0.4505  val=0.4607  AUC=0.7459
  * Best saved — AUC 0.7459
Epoch 06/25  train=0.4446  val=0.4583  AUC=0.7479
  * Best saved — AUC 0.7479
Epoch 07/25  train=0.4397  val=0.4594  AUC=0.7477
Epoch 08/25  train=0.4356  val=0.4617  AUC=0.7507
  * Best saved — AUC 0.7507
Epoch 09/25  train=0.4304  val=0.4492  AUC=0.7605
  * Best saved — AUC 0.7605
Epoch 10/25  train=0.4272  val=0.4509  AUC=0.7591
Epoch 11/25  train=0.4238  val=0.4542  AUC=0.7563


## 4. ECG-PT (supervised)

GPT-2 based decoder pretrained on ECG time-series. Originally unsupervised
(reconstruction loss). Adapted here for supervised classification by replacing
the causal-LM head with a sequence-classification head (last token -> linear -> 5).

Input signals are split into 36-sample patches and quantised to token IDs.

> **Note:** `Tconnector/ecg-pt` is not publicly available — falls back to plain GPT-2
> (same architecture, general-language weights). LoRA adapters use **r=16 alpha=32**, lr=2e-4.

In [6]:
from src.models.ecgpt import ECGPTClassifier

_probe = ECGPTClassifier(num_labels=5)
_probe.load()
_probe.apply_peft(use_dora=False)

_dummy_p = _probe.predict(np.random.randn(2, 1000).astype(np.float32))
assert _dummy_p.shape == (2, 5), f"Shape error: {_dummy_p.shape}"

_params = _probe.count_parameters()
assert _params['trainable'] / _params['total'] < 0.05, \
    f"Trainable fraction {_params['trainable']/_params['total']:.1%} exceeds 5%"

print(f"Shape OK : (2, 1000) -> {_dummy_p.shape}")
print(f"Params   : {_params['trainable']:,} / {_params['total']:,} = {_params['percentage']}")
del _probe

If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`).
  Falling back to gpt2 (GPT-2 architecture, general-language weights — NOT ECG-pretrained).
  To use the real ECG-PT weights, ensure the HF checkpoint is accessible (check your HF_TOKEN or the model visibility).


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 593,664 || all params: 125,037,312 || trainable%: 0.4748
Shape OK : (2, 1000) -> (2, 5)
Params   : 593,664 / 125,037,312 = 0.5%


In [7]:
from src.models.ecgpt import ECGPTClassifier

model_ecgpt_lora = ECGPTClassifier(num_labels=5)
model_ecgpt_lora.load()
model_ecgpt_lora.apply_peft(use_dora=False)

auc_ecgpt_lora, hist_ecgpt_lora = model_ecgpt_lora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'ecgpt_lora_r16',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_ecgpt_lora
import gc; gc.collect()
torch.cuda.empty_cache()

If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`).
  Falling back to gpt2 (GPT-2 architecture, general-language weights — NOT ECG-pretrained).
  To use the real ECG-PT weights, ensure the HF checkpoint is accessible (check your HF_TOKEN or the model visibility).


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 593,664 || all params: 125,037,312 || trainable%: 0.4748
  Tokenising signals...

 Experiment : ecgpt_lora_r16
 Device     : cuda
 Trainable  : 593,664  (0.5%)

Epoch 01/25  train=0.5435  val=0.5118  AUC=0.6233
  * Best saved — AUC 0.6233
Epoch 02/25  train=0.5120  val=0.5050  AUC=0.6273
  * Best saved — AUC 0.6273
Epoch 03/25  train=0.5083  val=0.5032  AUC=0.6289
  * Best saved — AUC 0.6289
Epoch 04/25  train=0.5058  val=0.5040  AUC=0.6317
  * Best saved — AUC 0.6317
Epoch 05/25  train=0.5044  val=0.5038  AUC=0.6314
Epoch 06/25  train=0.5018  val=0.5045  AUC=0.6325
  * Best saved — AUC 0.6325
Epoch 07/25  train=0.4997  val=0.5084  AUC=0.6266
Epoch 08/25  train=0.4981  val=0.5148  AUC=0.6251
  Early stopping (no AUC improvement for 2 epochs)

--------------------------------------------------
Profiling -- ecgpt_lora_r16
  Total time:       5.5min
  Avg epoch:        40.9s
  Peak GPU memory:  0.93 GB
  Trainable params: 593,664
  Total params:     125,037,312
 

In [8]:
from src.models.ecgpt import ECGPTClassifier

model_ecgpt_dora = ECGPTClassifier(num_labels=5)
model_ecgpt_dora.load()
model_ecgpt_dora.apply_peft(use_dora=True)

auc_ecgpt_dora, hist_ecgpt_dora = model_ecgpt_dora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'ecgpt_dora_r16',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_ecgpt_dora
import gc; gc.collect()
torch.cuda.empty_cache()

If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`).
  Falling back to gpt2 (GPT-2 architecture, general-language weights — NOT ECG-pretrained).
  To use the real ECG-PT weights, ensure the HF checkpoint is accessible (check your HF_TOKEN or the model visibility).


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 621,312 || all params: 125,064,960 || trainable%: 0.4968
  Tokenising signals...

 Experiment : ecgpt_dora_r16
 Device     : cuda
 Trainable  : 621,312  (0.5%)

Epoch 01/25  train=0.5374  val=0.5043  AUC=0.6324
  * Best saved — AUC 0.6324
Epoch 02/25  train=0.5115  val=0.5044  AUC=0.6313
Epoch 03/25  train=0.5075  val=0.5048  AUC=0.6328
  * Best saved — AUC 0.6328
Epoch 04/25  train=0.5058  val=0.5040  AUC=0.6331
  * Best saved — AUC 0.6331
Epoch 05/25  train=0.5038  val=0.5074  AUC=0.6326
Epoch 06/25  train=0.5026  val=0.5040  AUC=0.6338
  * Best saved — AUC 0.6338
Epoch 07/25  train=0.4997  val=0.5034  AUC=0.6306
Epoch 08/25  train=0.4970  val=0.5087  AUC=0.6278
  Early stopping (no AUC improvement for 2 epochs)

--------------------------------------------------
Profiling -- ecgpt_dora_r16
  Total time:       7.4min
  Avg epoch:        55.5s
  Peak GPU memory:  1.12 GB
  Trainable params: 621,312
  Total params:     125,064,960
  Trainable %:      0.5%
  Ch

## 3. HeartBERT

RoBERTa encoder pretrained on ECG-as-text. Weights are loaded from Google Drive
(`~/.cache/heartbert/` after first download). Input ECG signals are quantised into
**32-bin letter strings** (A–Z a–f) before tokenisation — up from 20 bins for better signal fidelity.

Each run:
1. Fresh `HeartBERTClassifier` instance; weights downloaded from Google Drive on first run
2. PEFT adapters attached (LoRA or DoRA, **r=16 alpha=32**)
3. Sanity check: output shape `(2, 5)`, trainable < 5 %
4. `fit()` — 25 epochs, AdamW lr=1.5e-4, BCEWithLogitsLoss, saves best checkpoint

In [9]:
from src.models.hubert_ecg import HuBERTECGClassifier

_probe = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
_probe.load()
_probe.apply_peft(use_dora=False)
_probe.to(device)

_dummy_x = torch.randn(2, 12, 1000, device=device)
with torch.no_grad():
    _out = _probe(_dummy_x)
assert _out.shape == (2, 5), f"Shape error: {_out.shape}"

_params = _probe.count_parameters()
assert _params['trainable'] / _params['total'] < 0.05, \
    f"Trainable fraction {_params['trainable']/_params['total']:.1%} exceeds 5%"

print(f"Shape OK : (2, 12, 1000) -> {_out.shape}")
print(f"Params   : {_params['trainable']:,} / {_params['total']:,} = {_params['percentage']}")
del _probe, _dummy_x, _out
torch.cuda.empty_cache()

Loading HuBERT-ECG (base) from Edoardo-BS/hubert-ecg-base ...
  Done. Hidden dim: 768
trainable params: 884,736 || all params: 94,013,829 || trainable%: 0.9411
Shape OK : (2, 12, 1000) -> torch.Size([2, 5])
Params   : 884,736 / 94,013,829 = 0.9%


In [10]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.training.train_peft import run_peft_experiment

model_hubert_lora = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model_hubert_lora.load()
model_hubert_lora.apply_peft(use_dora=False)
model_hubert_lora.to(device)

auc_hubert_lora, hist_hubert_lora, _ = run_peft_experiment(
    model_hubert_lora, train_ds_full, val_ds_full,
    experiment_name = 'hubert_ecg_lora_r16',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = RESULTS_PATH,
    num_workers     = CFG['training']['num_workers'],
)
del model_hubert_lora
import gc; gc.collect()
torch.cuda.empty_cache()

Loading HuBERT-ECG (base) from Edoardo-BS/hubert-ecg-base ...
  Done. Hidden dim: 768
trainable params: 884,736 || all params: 94,013,829 || trainable%: 0.9411

 Experiment : hubert_ecg_lora_r16
 Device     : cuda:0
 Trainable  : 884,736  (0.9%)

Epoch 01/25  lr=1.49e-04
  train_loss=0.3942  val_loss=0.3771
  AUC (macro): 0.8447
  Per-class AUC:
    NORM : 0.910  ##################
    MI   : 0.866  #################
    STTC : 0.893  #################
    CD   : 0.817  ################
    HYP  : 0.738  ##############
Adapter saved → D:\GitHub\biosignal-xai\results/hubert_ecg_lora_r16\best_adapter
Saved → D:\GitHub\biosignal-xai\results/hubert_ecg_lora_r16\best_adapter
  * Best saved — AUC 0.8447
Epoch 02/25  lr=1.48e-04
  train_loss=0.3469  val_loss=0.3663
  AUC (macro): 0.8589
  Per-class AUC:
    NORM : 0.910  ##################
    MI   : 0.882  #################
    STTC : 0.901  ##################
    CD   : 0.845  ################
    HYP  : 0.756  ###############
Adapter saved

In [11]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.training.train_peft import run_peft_experiment

model_hubert_dora = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model_hubert_dora.load()
model_hubert_dora.apply_peft(use_dora=True)
model_hubert_dora.to(device)

auc_hubert_dora, hist_hubert_dora, _ = run_peft_experiment(
    model_hubert_dora, train_ds_full, val_ds_full,
    experiment_name = 'hubert_ecg_dora_r16',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = RESULTS_PATH,
    num_workers     = CFG['training']['num_workers'],
)
del model_hubert_dora
import gc; gc.collect()
torch.cuda.empty_cache()

Loading HuBERT-ECG (base) from Edoardo-BS/hubert-ecg-base ...
  Done. Hidden dim: 768
trainable params: 912,384 || all params: 94,041,477 || trainable%: 0.9702

 Experiment : hubert_ecg_dora_r16
 Device     : cuda:0
 Trainable  : 912,384  (1.0%)

Epoch 01/25  lr=1.49e-04
  train_loss=0.3886  val_loss=0.4129
  AUC (macro): 0.8430
  Per-class AUC:
    NORM : 0.904  ##################
    MI   : 0.852  #################
    STTC : 0.895  #################
    CD   : 0.825  ################
    HYP  : 0.740  ##############
Adapter saved → D:\GitHub\biosignal-xai\results/hubert_ecg_dora_r16\best_adapter
Saved → D:\GitHub\biosignal-xai\results/hubert_ecg_dora_r16\best_adapter
  * Best saved — AUC 0.8430
Epoch 02/25  lr=1.48e-04
  train_loss=0.3444  val_loss=0.3850
  AUC (macro): 0.8495
  Per-class AUC:
    NORM : 0.896  #################
    MI   : 0.865  #################
    STTC : 0.896  #################
    CD   : 0.837  ################
    HYP  : 0.753  ###############
Adapter saved →

## 6. Summary

Validation macro-AUC across all six experiments. Trainable parameters and checkpoint
sizes reported from profiling.json files.

In [12]:
import json
from pathlib import Path

In [13]:
experiments = [
    ('heartbert_lora_r16',   auc_hb_lora),
    ('heartbert_dora_r16',   auc_hb_dora),
    ('ecgpt_lora_r16',       auc_ecgpt_lora),
    ('ecgpt_dora_r16',       auc_ecgpt_dora),
    # ('hubert_ecg_lora_r16',  auc_hubert_lora),
    # ('hubert_ecg_dora_r16',  auc_hubert_dora),
]

print(f"{'Experiment':<28}  {'Val AUC':>8}  {'Trainable':>12}  {'Ckpt MB':>8}")
print("-" * 62)

for exp_name, best_auc in experiments:
    prof_path = Path(RESULTS_PATH) / exp_name / 'profiling.json'
    if prof_path.exists():
        prof = json.loads(prof_path.read_text())
        trainable  = prof['trainable_params']
        ckpt_mb    = prof['checkpoint_size_mb']
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {trainable:>12,}  {ckpt_mb:>7.1f} MB")
    else:
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {'N/A':>12}  {'N/A':>8}")

print(f"\nBaseline FCN-Wang AUC (test set): 0.9225  (notebook 03)")
print(f"Full comparison plots → notebook 05")

Experiment                     Val AUC     Trainable   Ckpt MB
--------------------------------------------------------------
heartbert_lora_r16              0.7740       889,349      3.6 MB
heartbert_dora_r16              0.7605       898,565      3.6 MB
ecgpt_lora_r16                  0.6325       593,664      2.4 MB
ecgpt_dora_r16                  0.6338       621,312      2.5 MB

Baseline FCN-Wang AUC (test set): 0.9225  (notebook 03)
Full comparison plots → notebook 05


In [14]:
hubert_exp = [
    ('hubert_ecg_lora_r16',  auc_hubert_lora),
    ('hubert_ecg_dora_r16',  auc_hubert_dora),
]

print(f"{'Experiment':<28}  {'Val AUC':>8}  {'Trainable':>12}  {'Ckpt MB':>8}")
print("-" * 62)

for exp_name, best_auc in hubert_exp:
    prof_path = Path(RESULTS_PATH) / exp_name / 'profiling.json'
    if prof_path.exists():
        prof = json.loads(prof_path.read_text())
        trainable  = prof['trainable_params']
        ckpt_mb    = prof['checkpoint_size_mb']
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {trainable:>12,}  {ckpt_mb:>7.1f} MB")
    else:
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {'N/A':>12}  {'N/A':>8}")

print(f"\nBaseline FCN-Wang AUC (test set): 0.9225  (notebook 03)")
print(f"Full comparison plots → notebook 05")

Experiment                     Val AUC     Trainable   Ckpt MB
--------------------------------------------------------------
hubert_ecg_lora_r16             0.8675       884,736      3.5 MB
hubert_ecg_dora_r16             0.8700       912,384      3.7 MB

Baseline FCN-Wang AUC (test set): 0.9225  (notebook 03)
Full comparison plots → notebook 05
